In [0]:
df = spark.read.table("workspace.default.lc_analytical")
print("Rows:", df.count(), "| Cols:", len(df.columns))
df.printSchema()

Rows: 1345349 | Cols: 99
root
 |-- id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string (nullable = true)
 |-- delinq_2yrs: string (nullable = true)
 |-- earliest_cr_line: string (nullable = true)
 |-- fico_range_low: string (nullable = true)
 |-- fico_range_high: string (nullable = true)
 |-- inq_last_6mths: 

In [0]:
from pyspark.sql.functions import col, year, month, to_date, months_between, current_date

df = df.withColumn("issue_year", year(col("issue_d")))
df = df.withColumn("issue_month", month(col("issue_d")))

In [0]:
df = df.withColumn(
    "credit_history_months",
    months_between(col("issue_d"), col("earliest_cr_line"))
)

In [0]:
df = spark.read.table("workspace.default.lc_analytical")
print("Cols:", len(df.columns))
print(df.columns)

Cols: 99
['id', 'loan_amnt', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', '

In [0]:
df_raw = spark.read.csv(
    "/Volumes/workspace/default/lending_club/Lending_Club_Accepted.csv",
    header=True, inferSchema=True
).select("id", "issue_d")

df = df.join(df_raw, on="id", how="left")
df.select("id", "issue_d", "earliest_cr_line").show(5)

+--------+--------+----------------+
|      id| issue_d|earliest_cr_line|
+--------+--------+----------------+
|98347436|Feb-2017|        Aug-2010|
|96891368|Feb-2017|        Oct-2010|
|98014601|Feb-2017|        Aug-1972|
|98155033|Feb-2017|        Oct-2005|
|97380238|Feb-2017|        Nov-2013|
+--------+--------+----------------+
only showing top 5 rows


In [0]:
df = spark.read.table("workspace.default.lc_analytical")
df_raw = spark.read.csv(
    "/Volumes/workspace/default/lending_club/Lending_Club_Accepted.csv",
    header=True, inferSchema=True
).select("id", "issue_d")
df = df.join(df_raw, on="id", how="left")

In [0]:
from pyspark.sql.functions import col, year, month, when, to_date, months_between

# Only attempt parsing if the string matches the MMM-yyyy pattern, else null
df = df.withColumn(
    "issue_d",
    when(col("issue_d").rlike("^[A-Za-z]{3}-\\d{4}$"),
         to_date(col("issue_d"), "MMM-yyyy"))
)
df = df.withColumn(
    "earliest_cr_line",
    when(col("earliest_cr_line").rlike("^[A-Za-z]{3}-\\d{4}$"),
         to_date(col("earliest_cr_line"), "MMM-yyyy"))
)

df = df.withColumn(
    "credit_history_months",
    months_between(col("issue_d"), col("earliest_cr_line"))
)
df = df.withColumn("issue_year", year(col("issue_d")))
df = df.withColumn("issue_month", month(col("issue_d")))

df.select("issue_d", "earliest_cr_line", "credit_history_months").show(5)
df.select("credit_history_months").summary("min", "25%", "50%", "75%", "max").show()

+----------+----------------+---------------------+
|   issue_d|earliest_cr_line|credit_history_months|
+----------+----------------+---------------------+
|2017-02-01|      2010-08-01|                 78.0|
|2017-02-01|      2010-10-01|                 76.0|
|2017-02-01|      1972-08-01|                534.0|
|2017-02-01|      2005-10-01|                136.0|
|2017-02-01|      2013-11-01|                 39.0|
+----------+----------------+---------------------+
only showing top 5 rows
+-------+---------------------+
|summary|credit_history_months|
+-------+---------------------+
|    min|                 36.0|
|    25%|                135.0|
|    50%|                177.0|
|    75%|                240.0|
|    max|                999.0|
+-------+---------------------+



In [0]:
from pyspark.sql.functions import col

for c in ["annual_inc", "dti", "revol_util", "loan_amnt", "int_rate"]:
    df = df.withColumn(c, col(c).cast("double"))

df.select("annual_inc", "dti", "revol_util", "loan_amnt", "int_rate").printSchema()

root
 |-- annual_inc: double (nullable = true)
 |-- dti: double (nullable = true)
 |-- revol_util: double (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- int_rate: double (nullable = true)



In [0]:
from pyspark.sql.functions import col, expr, year, month, when, to_date, months_between

# 1. Fresh load from the saved table
df = spark.read.table("workspace.default.lc_analytical")

# 2. Join issue_d back from raw CSV
df_raw = spark.read.csv(
    "/Volumes/workspace/default/lending_club/Lending_Club_Accepted.csv",
    header=True, inferSchema=True
).select("id", "issue_d")
df = df.join(df_raw, on="id", how="left")

# 3. try_cast numeric columns (handles 'Karen', '326xx', etc.)
for c in ["annual_inc", "dti", "revol_util", "loan_amnt", "int_rate"]:
    df = df.withColumn(c, expr(f"try_cast({c} AS DOUBLE)"))

# 4. Parse dates via regex guard (rejects '7.42' etc.)
df = df.withColumn(
    "issue_d",
    when(col("issue_d").rlike("^[A-Za-z]{3}-\\d{4}$"),
         to_date(col("issue_d"), "MMM-yyyy"))
)
df = df.withColumn(
    "earliest_cr_line",
    when(col("earliest_cr_line").rlike("^[A-Za-z]{3}-\\d{4}$"),
         to_date(col("earliest_cr_line"), "MMM-yyyy"))
)

# 5. Derived features
df = df.withColumn("credit_history_months", months_between(col("issue_d"), col("earliest_cr_line")))
df = df.withColumn("issue_year", year(col("issue_d")))
df = df.withColumn("issue_month", month(col("issue_d")))

print("Rebuilt df. Columns:", len(df.columns))
df.select("annual_inc", "dti", "revol_util", "loan_amnt", "int_rate", "issue_d", "earliest_cr_line", "credit_history_months").printSchema()

Rebuilt df. Columns: 103
root
 |-- annual_inc: double (nullable = true)
 |-- dti: double (nullable = true)
 |-- revol_util: double (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- issue_d: date (nullable = true)
 |-- earliest_cr_line: date (nullable = true)
 |-- credit_history_months: double (nullable = true)



In [0]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c + "_nulls")
    for c in ["annual_inc", "dti", "revol_util", "loan_amnt", "int_rate", "issue_d", "earliest_cr_line"]
]).show()

+----------------+---------+----------------+---------------+--------------+-------------+----------------------+
|annual_inc_nulls|dti_nulls|revol_util_nulls|loan_amnt_nulls|int_rate_nulls|issue_d_nulls|earliest_cr_line_nulls|
+----------------+---------+----------------+---------------+--------------+-------------+----------------------+
|               0|      600|             912|              0|             0|            0|                   226|
+----------------+---------+----------------+---------------+--------------+-------------+----------------------+



In [0]:
from pyspark.ml.feature import QuantileDiscretizer

qd_inc = QuantileDiscretizer(inputCol="annual_inc", outputCol="inc_bin", numBuckets=5, handleInvalid="keep")
df = qd_inc.fit(df).transform(df)

qd_dti = QuantileDiscretizer(inputCol="dti", outputCol="dti_bin", numBuckets=5, handleInvalid="keep")
df = qd_dti.fit(df).transform(df)

qd_util = QuantileDiscretizer(inputCol="revol_util", outputCol="util_bin", numBuckets=5, handleInvalid="keep")
df = qd_util.fit(df).transform(df)

print("Bins added.")

Bins added.


In [0]:
from pyspark.sql.functions import avg, count

for c in ["inc_bin", "dti_bin", "util_bin"]:
    print(f"\n--- {c} ---")
    df.groupBy(c).agg(
        avg("is_bad").alias("default_rate"),
        count("*").alias("n")
    ).orderBy(c).show()


--- inc_bin ---
+-------+-------------------+------+
|inc_bin|       default_rate|     n|
+-------+-------------------+------+
|    0.0|0.23535242077549992|258285|
|    1.0|0.21697479776091805|278507|
|    2.0|0.20326937542803894|267207|
|    3.0|0.18528183716075156|264408|
|    4.0|0.15915245791537577|276942|
+-------+-------------------+------+


--- dti_bin ---
+-------+-------------------+------+
|dti_bin|       default_rate|     n|
+-------+-------------------+------+
|   NULL|              0.175|   600|
|    0.0|0.14959967855229478|268784|
|    1.0| 0.1676295844066888|268628|
|    2.0|0.19090973303939537|268991|
|    3.0|0.21944776213810785|268761|
|    4.0|0.27049353636144446|269585|
+-------+-------------------+------+


--- util_bin ---
+--------+-------------------+------+
|util_bin|       default_rate|     n|
+--------+-------------------+------+
|    NULL|0.21162280701754385|   912|
|     0.0| 0.1579870093614487|268655|
|     1.0|0.18972284823067623|268012|
|     2.0| 0.20

In [0]:
from pyspark.sql.functions import when, col
from pyspark.ml.feature import StringIndexer

# Ordinal: grade → integer
df = df.withColumn(
    "grade_num",
    when(col("grade") == "A", 1)
    .when(col("grade") == "B", 2)
    .when(col("grade") == "C", 3)
    .when(col("grade") == "D", 4)
    .when(col("grade") == "E", 5)
    .when(col("grade") == "F", 6)
    .when(col("grade") == "G", 7)
)

# Nominal: index-encode
for c in ["home_ownership", "purpose", "verification_status"]:
    indexer = StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    df = indexer.fit(df).transform(df)

print("Encoded.")
df.select("grade", "grade_num", "home_ownership", "home_ownership_idx").show(5)

Encoded.
+-----+---------+--------------+------------------+
|grade|grade_num|home_ownership|home_ownership_idx|
+-----+---------+--------------+------------------+
|    C|        3|      MORTGAGE|               0.0|
|    C|        3|      MORTGAGE|               0.0|
|    F|        6|          RENT|               1.0|
|    B|        2|          RENT|               1.0|
|    C|        3|      MORTGAGE|               0.0|
+-----+---------+--------------+------------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import avg, count
df.groupBy("grade_num").agg(avg("is_bad").alias("default_rate"), count("*").alias("n")).orderBy("grade_num").show()

+---------+--------------------+------+
|grade_num|        default_rate|     n|
+---------+--------------------+------+
|        1|0.060426636040749486|235095|
|        2| 0.13386480355037722|392747|
|        3|  0.2244127494799499|381694|
|        4|  0.3038673208403411|200966|
|        5|  0.3848231827111984| 93656|
|        6| 0.45204154839514643| 32059|
|        7| 0.49934296977660975|  9132|
+---------+--------------------+------+



In [0]:
feature_cols = [
    # Target
    "is_bad",
    # Loan terms
    "loan_amnt", "int_rate", "term_months", "grade_num",
    # Borrower financials (binned)
    "inc_bin", "dti_bin", "util_bin",
    # Credit history
    "credit_history_months", "has_prior_delinq", "has_public_record",
    # Encoded categoricals
    "home_ownership_idx", "purpose_idx", "verification_status_idx",
    # Vintage controls
    "issue_year",
    # Keep raw for reference / WoE tomorrow
    "annual_inc", "dti", "revol_util", "grade", "purpose"
]

df_features = df.select(*feature_cols)
df_features.printSchema()
df_features.show(5)

root
 |-- is_bad: integer (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- term_months: integer (nullable = true)
 |-- grade_num: integer (nullable = true)
 |-- inc_bin: double (nullable = true)
 |-- dti_bin: double (nullable = true)
 |-- util_bin: double (nullable = true)
 |-- credit_history_months: double (nullable = true)
 |-- has_prior_delinq: integer (nullable = true)
 |-- has_public_record: integer (nullable = true)
 |-- home_ownership_idx: double (nullable = false)
 |-- purpose_idx: double (nullable = false)
 |-- verification_status_idx: double (nullable = false)
 |-- issue_year: integer (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- dti: double (nullable = true)
 |-- revol_util: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- purpose: string (nullable = true)

+------+---------+--------+-----------+---------+-------+-------+--------+---------------------+----------------+----------------

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show()
spark.sql("SHOW VOLUMES IN workspace.default").show()

+--------+-------------+-----------+
|database|    tableName|isTemporary|
+--------+-------------+-----------+
| default|lc_analytical|      false|
+--------+-------------+-----------+

+--------+------------+
|database| volume_name|
+--------+------------+
| default|lending_club|
+--------+------------+



In [0]:
df_features.write.mode("overwrite").saveAsTable("workspace.default.lc_features")

In [0]:
df_check = spark.table("workspace.default.lc_features")
print("Rows:", df_check.count(), "| Cols:", len(df_check.columns))
df_check.printSchema()

Rows: 1345349 | Cols: 20
root
 |-- is_bad: integer (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- term_months: integer (nullable = true)
 |-- grade_num: integer (nullable = true)
 |-- inc_bin: double (nullable = true)
 |-- dti_bin: double (nullable = true)
 |-- util_bin: double (nullable = true)
 |-- credit_history_months: double (nullable = true)
 |-- has_prior_delinq: integer (nullable = true)
 |-- has_public_record: integer (nullable = true)
 |-- home_ownership_idx: double (nullable = true)
 |-- purpose_idx: double (nullable = true)
 |-- verification_status_idx: double (nullable = true)
 |-- issue_year: integer (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- dti: double (nullable = true)
 |-- revol_util: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- purpose: string (nullable = true)

